In [ ]:
from matplotlib import pyplot as plt
import seaborn as sns

In [ ]:
import cartopy.crs as ccrs
import cartopy.feature as cfeature

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr

from sklearn.metrics import root_mean_squared_error

sys.path.append("../..")
from src.data.dataset_utils import exclude_boundary_nodes
from src.utils.plot_functions import plot_overall_scores, plot_binned_scores, plot_spatial_grids, spatial_grid_from_dataarray

GROUND_TRUTH_ZARR = "/work/ka1176/shared_data/2025-wwm-emulator-schism/processed_RUN04b_JUWELS_wpile.zarr"

# produced by notebooks/paper-plots/compute_experiment_metrics.py
METRICS_DIR = Path("metrics")

In [ ]:
PAPER_WIDE = (7.1, 3.5)
PAPER_NARROW = (3.5, 2.8)

In [ ]:
sns.set_context("paper")
sns.set_style("whitegrid")

In [ ]:
def parse_bin_column(bin_series: pd.Series) -> pd.Series:
    """Turn the 'bin' column of a binned-metrics CSV back into pd.Interval objects.

    compute_binned_scores/compute_binned_change_scores return 'bin' as a pd.Interval
    (or the string 'overall'), which round-trips through CSV as a plain string like
    '(0.0, 0.5]'. plot_binned_scores expects real pd.Interval objects (for .mid), so
    we parse them back here after loading the CSV.
    """

    def _parse(value):
        if value == "overall":
            return value
        left, right = value.strip("()[]").split(", ")
        return pd.Interval(float(left), float(right), closed="right")

    return bin_series.apply(_parse)

## Read pre-computed metrics

Use `compute_experiment_metrics.py` to pre-compute metrics. 

In [ ]:
summary_df = pd.read_csv(METRICS_DIR / "summary_metrics.csv")
summary_df.round(decimals=3) # mm is already stretching what can be meaningfully assessed in terms of Hs

In [ ]:
binned_value_df = pd.read_csv(METRICS_DIR / "binned_value_metrics.csv").query("variable == 'sigWaveHeight'")
binned_value_df["bin"] = parse_bin_column(binned_value_df["bin"])

In [ ]:
binned_change_df = pd.read_csv(METRICS_DIR / "binned_change_metrics.csv").query("variable == 'sigWaveHeight'")
binned_change_df["bin"] = parse_bin_column(binned_change_df["bin"])

## Single-step RMSE

### Single-target: Significant wave height

Show RMSE and bias for selected reference experiments.

In [ ]:
keep_experiments = ["swh-mlp", "swh-lstm-6", "swh-lstm-48", "swh-gat-500", "swh-gat-vn-500", "swh-gat-lstm-vn-6-50", "swh-gat-lstm-vn-6-500", "swh-gat-lstm-vn-48-50"]

In [ ]:
# define architecture priority for sorting
arch_priority = {
    "mlp": 0,
    "lstm": 1,
    "gat": 2,        # Graph (must check before gat-vn / gat-lstm-vn, see note below)
    "gat-vn": 3,      # Graph-VN
    "gat-lstm-vn": 4, # Graph-LSTM
}

def get_arch_key(experiment):
    # order matters: check longest/most-specific patterns first
    if "gat-lstm-vn" in experiment:
        return arch_priority["gat-lstm-vn"]
    if "gat-vn" in experiment:
        return arch_priority["gat-vn"]
    if "gat" in experiment:
        return arch_priority["gat"]
    if "lstm" in experiment:
        return arch_priority["lstm"]
    if "mlp" in experiment:
        return arch_priority["mlp"]
    return 99  # fallback, shouldn't hit if keep_experiments is well-formed

hue_order = sorted(keep_experiments, key=get_arch_key)

new_labels = ["MLP", "LSTM (6 h)", "LSTM (48 h)", "Graph (w/o VN) (500 nodes)", "Graph (500 nodes)", "GraphLSTM (6 h, 50 nodes)", "GraphLSTM (6 h, 500 nodes)", "GraphLSTM (48 h, 50 nodes)"]
new_compact_labels = ["MLP", "LSTM (6 h)", "LSTM (48 h)", "Graph (w/o VN)\n(500 nodes)", "Graph (500 nodes)", "GraphLSTM\n(6 h, 50 nodes)", "GraphLSTM\n(6 h, 500 nodes)", "GraphLSTM\n(48 h, 50 nodes)"]

plot_colors = {
     'swh-mlp': '#0072B2',
     'swh-lstm-6': '#b27b00',
     'swh-lstm-48': '#ffd066',
     'swh-gat-500': '#D55E00',
     'swh-gat-vn-500': '#009E73',
     'swh-gat-lstm-vn-6-50': '#81315e',
     'swh-gat-lstm-vn-6-500': '#bc508c',
     'swh-gat-lstm-vn-48-50': '#d590b6',
}

In [ ]:
binned_change_df["bin_str"] = binned_change_df["bin"].astype(str)
binned_change_df["bin_str"] = binned_change_df["bin_str"].replace({"(-0.5, 0.0]":"(-0.5, 0.5]", "(0.0, 0.5]":"(-0.5, 0.5]"})

In [ ]:
# Paper figure: RMSE (swh) per bin, Bias (dswh) per bin
fig = plt.figure(figsize=PAPER_WIDE)
gs = fig.add_gridspec(2, 2, width_ratios=[3.5, 1])

ax0 = fig.add_subplot(gs[0, 0])
ax1 = fig.add_subplot(gs[1, 0])

ax = [ax0, ax1]

sns.barplot(binned_value_df.query("experiment in @keep_experiments").query("bin != 'overall'"),
            x="bin", y="rmse", hue="experiment", hue_order=hue_order, palette=plot_colors,
            linewidth=1, edgecolor="w", ax=ax[0], legend=True,
           )

ax[0].set_xlabel("True significant wave height (m)")
ax[0].set_ylabel("RMSE (m)")

ax[0].set_xticks([0, 1, 2, 3, 4])
ax[0].set_xticklabels(["[0, 0.5]", "(0.5, 1.0]", "(1.0, 2.0]", "(2.0, 4.0]", "> 4.0"])

ax[0].legend().set_visible(False)

ax[0].set_yticks([0, 0.05, 0.1, 0.15, 0.2, 0.25])

sns.barplot(binned_change_df.query("experiment in @keep_experiments").query("bin_str != 'overall'"),
            x="bin_str", y="bias", hue="experiment", hue_order=hue_order, palette=plot_colors,
            linewidth=1, edgecolor="w", errorbar=None, ax=ax[1], legend=False, 
           )

ax[1].set_xlabel("True change in significant wave height (m)")
ax[1].set_ylabel("Bias (m)")

ax[1].set_xticks([0, 1, 2, 3, 4])
ax[1].set_xticklabels(["(-2.0, -1.0]", "(-1.0, -0.5]", "(-0.5, 0.5]", "(0.5, 1.0]", "(1.0, 2.0]"])

ax[1].set_yticks([-0.6, -0.4, -0.2, 0, 0.2])

handles = ax[0].get_legend_handles_labels()[0]
fig.legend(handles, new_compact_labels, ncol=1, loc="center right", title="WindWaveNet\narchitecture")

ax[1].hlines(0,-1,6, color="C7", ls="-", lw=0.7)
ax[1].set_xlim(-0.5, 4.5)

# subplot labels
ax[0].text(-0.12, 1, "A", transform=ax[0].transAxes, ha="left", va="top", fontweight="bold")
ax[1].text(-0.12, 1, "B", transform=ax[1].transAxes, ha="left", va="top", fontweight="bold")

fig.tight_layout()
plt.savefig("paper_plots/swh_dswh_bins.png", dpi=200, bbox_inches="tight")

Single plots for reference, not included in the manuscript

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=PAPER_WIDE)

sns.barplot(binned_value_df.query("experiment in @keep_experiments").query("bin != 'overall'"),
            x="bin", y="rmse", hue="experiment", hue_order=hue_order, palette=plot_colors,
            linewidth=1, edgecolor="w",
           )

ax.set_xlabel("True significant wave height (m)")
ax.set_ylabel("RMSE (m)")

ax.set_xticks([0, 1, 2, 3, 4])
ax.set_xticklabels(["< 0.5", "0.5 - 1.0", "1.0 - 2.0", "2.0 - 4.0", "> 4.0"])

handles = ax.get_legend_handles_labels()[0]
ax.legend(handles, new_labels)

fig.tight_layout()

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=PAPER_WIDE)

sns.barplot(binned_change_df.query("experiment in @keep_experiments").query("bin != 'overall'"),
            x="bin", y="bias", hue="experiment", hue_order=hue_order, palette=plot_colors,
            linewidth=1, edgecolor="w",
           )

ax.set_xlabel("True change in significant wave height (m)")
ax.set_ylabel("Bias (m)")

ax.set_xticks([0, 1, 2, 3, 4, 5])
ax.set_xticklabels(["-2.0 - 1.0", "-1.0 - 0.5", "-0.5 - 0", "0 - 0.5", "0.5 - 1.0", "1.0 - 2.0"])


handles = ax.get_legend_handles_labels()[0]
ax.legend(handles, new_labels, ncols=2)

fig.tight_layout()

### Multi-Target

Show RMSE per bin of target variable for all target variables, first in separate panels for reference, and finally in a combined panel to include in the paper.

In [ ]:
binned_value_df = pd.read_csv(METRICS_DIR / "binned_value_metrics.csv") # read again since we selected only swh above

In [ ]:
keep_multi_experiments = ["multi-lstm-6", "multi-lstm-48", "multi-gat-vn-500", "multi-gat-lstm-vn-6-500"] 
hue_order = sorted(keep_multi_experiments, key=get_arch_key)
new_labels = ["LSTM (6 h)", "LSTM (48 h)", "Graph (500 nodes)", "GraphLSTM (6 h, 500 nodes)"]

In [ ]:
plot_colors = {
 'multi-lstm-6': '#b27b00',
 'multi-lstm-48': '#ffd066',
 'multi-gat-vn-500': '#009E73',
 'multi-gat-lstm-vn-6-500': '#bc508c',
}

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=PAPER_WIDE)

sns.barplot(binned_value_df.query("experiment in @keep_multi_experiments").query("variable == 'sigWaveHeight'").query("bin != 'overall'"),
            x="bin", y="rmse", hue="experiment", hue_order=hue_order, palette=plot_colors,
            linewidth=1, edgecolor="w",
           )

ax.set_xlabel("True significant wave height (m)")
ax.set_ylabel("RMSE (m)")

ax.set_xticks([0, 1, 2, 3, 4])
ax.set_xticklabels(["< 0.5", "0.5 - 1.0", "1.0 - 2.0", "2.0 - 4.0", "> 4.0"])

handles = ax.get_legend_handles_labels()[0]
ax.legend(handles, new_labels)

fig.tight_layout()

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=PAPER_WIDE)

sns.barplot(binned_value_df.query("experiment in @keep_multi_experiments").query("variable == 'meanWavePeriod'").query("bin != 'overall'"),
            x="bin", y="rmse", hue="experiment", hue_order=hue_order, palette=plot_colors,
            linewidth=1, edgecolor="w",
           )

ax.set_xlabel("True mean wave period (s)")
ax.set_ylabel("RMSE (s)")

ax.set_xticks([0, 1, 2, 3])
ax.set_xticklabels(["< 3", "3 - 5", "5 - 7", "> 7"])

handles = ax.get_legend_handles_labels()[0]
ax.legend(handles, new_labels)

fig.tight_layout()

In [ ]:
mwd_values = binned_value_df.query("experiment in @keep_multi_experiments").query("variable == 'meanWaveDirection'").query("bin != 'overall'")
mwd_values = mwd_values.replace({"(0, 45]":45/2, "(45, 90]":45+45/2, "(90, 135]":90+45/2, "(135, 180]":135+45/2, "(180, 225]":180+45/2, "(225, 270]":225+45/2, "(270, 315]":270+45/2, "(315, 360]":315+45/2})

In [ ]:
mwd_values.head()

In [ ]:
for i, key in enumerate(keep_multi_experiments):
    plt.polar(np.deg2rad(mwd_values.query("experiment==@key").bin.values), mwd_values.query("experiment==@key").rmse.values, marker="o", label=key, color=plot_colors[key])

plt.legend()
plt.gca().set_theta_direction(-1)
plt.gca().set_theta_offset(+np.pi/2)

In [ ]:
# Paper figure: RMSE per bin of the target variable for the single-step, multi-target WindWaveNet

fig = plt.figure(figsize=PAPER_WIDE)
gs = fig.add_gridspec(2, 2, width_ratios=[2, 1])


ax0 = fig.add_subplot(gs[0, 0])
ax1 = fig.add_subplot(gs[1, 0])
ax2 = fig.add_subplot(gs[:, 1], projection="polar")

# significant wave height
sns.barplot(binned_value_df.query("experiment in @keep_multi_experiments").query("variable == 'sigWaveHeight'").query("bin != 'overall'"),
            x="bin", y="rmse", hue="experiment", hue_order=hue_order, ax=ax0, legend=False, palette=plot_colors,
            linewidth=1, edgecolor="w",
           )

ax0.set_xlabel("True significant wave height (m)")
ax0.set_ylabel("RMSE (m)")

ax0.set_xticks([0, 1, 2, 3, 4])
#ax0.set_xticklabels(["< 0.5", "0.5 - 1.0", "1.0 - 2.0", "2.0 - 4.0", "> 4.0"])
ax0.set_xticklabels(["[0, 0.5]", "(0.5, 1.0]", "(1.0, 2.0]", "(2.0, 4.0]", "> 4.0"])

#handles = ax0.get_legend_handles_labels()[0]
#ax0.legend(handles, new_labels)

# mean wave period
sns.barplot(binned_value_df.query("experiment in @keep_multi_experiments").query("variable == 'meanWavePeriod'").query("bin != 'overall'"),
            x="bin", y="rmse", hue="experiment", hue_order=hue_order, ax=ax1, legend=False, palette=plot_colors,
            linewidth=1, edgecolor="w",
           )

ax1.set_xlabel("True mean wave period (s)")
ax1.set_ylabel("RMSE (s)")
ax1.set_xticks([0, 1, 2, 3])
ax1.set_xticklabels(["[0, 3)", "(3, 5]", "[5, 7]", "> 7"])

# mean wave direction
markers = ["o", "s", "*", "v"]
for i, key in enumerate(keep_multi_experiments):
    ax2.plot(np.deg2rad(mwd_values.query("experiment==@key").bin.values), mwd_values.query("experiment==@key").rmse.values, marker=markers[i], label=key, color=plot_colors[key])
ax2.set_theta_direction(-1)
ax2.set_theta_offset(np.pi/2)
ax2.text(0, -0.22, "RMSE of mean wave direction", transform=ax2.transAxes)

# subplot labels
for ax, label in zip([ax0, ax1, ax2], ["A", "B", "C"]):
    ax.text(-0.1, 1.05, label, transform=ax.transAxes,
            fontsize=9, fontweight="bold", va="bottom", ha="right")

# single shared legend for the whole figure, built from ax0's (now hidden) handles
handles, _ = ax2.get_legend_handles_labels()
fig.legend(handles, new_labels, loc="upper center", bbox_to_anchor=(0.5, 1.05),
           ncol=len(new_labels), frameon=False)

fig.tight_layout()
plt.savefig("paper_plots/multi_rmse_all.png", dpi=200, bbox_inches="tight")


### Spatial analysis

#### Read in pre-computed spatial metrics

Run `compute_spatial_metrics.py` to generate the spatial metrics for the multi-target experiments.

In [ ]:
skeys = ["mlp", "lstm-6", "lstm-48", "gat-500", "gat-vn-500", "gat-lstm-vn-6-500"]
slabelkeys = ["MLP", "LSTM (6 h)", "LSTM (48 h)", "Graph w/o VN\n(500 nodes)", "Graph (500 nodes)", "Graph-LSTM\n(6 h, 500 nodes)"]
sdict = {}

for key, lkey in zip(skeys, slabelkeys):
    sdict[key] = xr.open_dataset(f"./spatial_metrics/swh-{key}_spatial_metrics.nc")

In [ ]:
grids = {key: spatial_grid_from_dataarray(sdict[key].sel(season="winter")["rmse"]) for key in skeys}

In [ ]:
gt_grids = {
    "summer": spatial_grid_from_dataarray(sdict["mlp"]["ground_truth_mean"].sel(season="summer")),
    "winter": spatial_grid_from_dataarray(sdict["mlp"]["ground_truth_mean"].sel(season="winter")),
}

#### Single-target runs

In [ ]:
fig, ax = plt.subplots(2, 4, sharex=True, sharey=True, figsize=PAPER_WIDE, subplot_kw={"projection": ccrs.Orthographic(central_longitude=4.5, central_latitude=53.5)})

# ground truth
gpcm = ax[0,0].pcolormesh(gt_grids["winter"]["xi"], gt_grids["winter"]["yi"], gt_grids["winter"]["zi"],
                         cmap="viridis", vmin=0, vmax=2., transform=ccrs.UTM(32))


gcb = plt.colorbar(gpcm, ax=ax[1, 0], extend="max", orientation="horizontal", pad=-.01, shrink=0.8, anchor=(0.5, 7), label="Hs (m)")
gcb.set_ticks([0, 0.5, 1, 1.5, 2])

# MLP RMSE etc
vmax=0.1
ax[0,1].pcolormesh(grids["mlp"]["xi"], grids["mlp"]["yi"], grids["mlp"]["zi"],
                   cmap="YlOrRd", vmin=0, vmax=vmax, transform=ccrs.UTM(32),
                  )

ax[0,2].pcolormesh(grids["lstm-6"]["xi"], grids["lstm-6"]["yi"], grids["lstm-6"]["zi"],
                   cmap="YlOrRd", vmin=0, vmax=vmax, transform=ccrs.UTM(32),
                  )

ax[0,3].pcolormesh(grids["lstm-48"]["xi"], grids["lstm-48"]["yi"], grids["lstm-48"]["zi"],
                   cmap="YlOrRd", vmin=0, vmax=vmax, transform=ccrs.UTM(32),
                  )

ax[1,1].pcolormesh(grids["gat-500"]["xi"], grids["gat-500"]["yi"], grids["gat-500"]["zi"],
                   cmap="YlOrRd", vmin=0, vmax=vmax, transform=ccrs.UTM(32),
                  )

ax[1,2].pcolormesh(grids["gat-vn-500"]["xi"], grids["gat-vn-500"]["yi"], grids["gat-vn-500"]["zi"],
                   cmap="YlOrRd", vmin=0, vmax=vmax, transform=ccrs.UTM(32),
                  )

pcm = ax[1,3].pcolormesh(grids["gat-lstm-vn-6-500"]["xi"], grids["gat-lstm-vn-6-500"]["yi"], grids["gat-lstm-vn-6-500"]["zi"],
                   cmap="YlOrRd", vmin=0, vmax=vmax, transform=ccrs.UTM(32),
                  )

pcb = plt.colorbar(pcm, ax=ax[1, 0], extend="max", orientation="horizontal", shrink=0.8, label="RMSE (m)", anchor=(0.5, 2), panchor=False)
ax[1,0].set_visible(False)


# extent and layout
ax[0,0].set_extent([-1.5, 9.5, 49.5, 56])

[axx.add_feature(cfeature.LAND, facecolor="lightgray", zorder=2) for axx in ax.flatten()]
[axx.add_feature(cfeature.BORDERS, linewidth=0.5, edgecolor="black", zorder=3) for axx in ax.flatten()]
[axx.coastlines(zorder=2, lw=0.5) for axx in ax.flatten()]

ax[0,0].text(0, 1, "A ", transform=ax[0,0].transAxes, ha="right", va="top", fontweight="bold")
ax[0,1].text(0, 1, "B ", transform=ax[0,1].transAxes, ha="right", va="top", fontweight="bold")
ax[0,2].text(0, 1, "C ", transform=ax[0,2].transAxes, ha="right", va="top", fontweight="bold")
ax[0,3].text(0, 1, "D ", transform=ax[0,3].transAxes, ha="right", va="top", fontweight="bold")
ax[1,1].text(0, 1, "E ", transform=ax[1,1].transAxes, ha="right", va="top", fontweight="bold")
ax[1,2].text(0, 1, "F ", transform=ax[1,2].transAxes, ha="right", va="top", fontweight="bold")
ax[1,3].text(0, 1, "G ", transform=ax[1,3].transAxes, ha="right", va="top", fontweight="bold")

ax[0,0].text(.95, 0.05, "Ground truth", transform=ax[0,0].transAxes, ha="right", va="bottom", fontsize=8, bbox={"fc":"w"})
ax[0,1].text(.95, 0.05, "MLP", transform=ax[0,1].transAxes, ha="right", va="bottom", fontsize=8, bbox={"fc":"w"})
ax[0,2].text(.95, 0.05, "LSTM (6 h)", transform=ax[0,2].transAxes, ha="right", va="bottom", fontsize=8, bbox={"fc":"w"})
ax[0,3].text(.95, 0.05, "LSTM (48 h)", transform=ax[0,3].transAxes, ha="right", va="bottom", fontsize=8, bbox={"fc":"w"})
ax[1,1].text(.95, 0.05, "Graph w/o VN", transform=ax[1,1].transAxes, ha="right", va="bottom", fontsize=8, bbox={"fc":"w"})
ax[1,2].text(.95, 0.05, "Graph", transform=ax[1,2].transAxes, ha="right", va="bottom", fontsize=8, bbox={"fc":"w"})
ax[1,3].text(.95, 0.05, "GraphLSTM (6 h)", transform=ax[1,3].transAxes, ha="right", va="bottom", fontsize=8, bbox={"fc":"w"})

plt.subplots_adjust(wspace=0.15, hspace=0.05)

plt.savefig("paper_plots/spatial_rmse_winter_with_ground_truth.png", dpi=220, bbox_inches="tight")

In [ ]:
fig, ax = plt.subplots(2, 4, sharex=True, sharey=True, figsize=PAPER_WIDE, subplot_kw={"projection": ccrs.Orthographic(central_longitude=4.5, central_latitude=53.5)})

# ground truth
gpcm = ax[0,0].pcolormesh(gt_grids["summer"]["xi"], gt_grids["summer"]["yi"], gt_grids["summer"]["zi"],
                         cmap="viridis", vmin=0, vmax=2., transform=ccrs.UTM(32))


gcb = plt.colorbar(gpcm, ax=ax[1, 0], extend="max", orientation="horizontal", pad=-.01, shrink=0.8, anchor=(0.5, 7), label="Hs (m)")
gcb.set_ticks([0, 0.5, 1, 1.5, 2])

# MLP RMSE etc
grids = {key: spatial_grid_from_dataarray(sdict[key].sel(season="summer")["rmse"]) for key in skeys}

vmax=0.1
ax[0,1].pcolormesh(grids["mlp"]["xi"], grids["mlp"]["yi"], grids["mlp"]["zi"],
                   cmap="YlOrRd", vmin=0, vmax=vmax, transform=ccrs.UTM(32),
                  )

ax[0,2].pcolormesh(grids["lstm-6"]["xi"], grids["lstm-6"]["yi"], grids["lstm-6"]["zi"],
                   cmap="YlOrRd", vmin=0, vmax=vmax, transform=ccrs.UTM(32),
                  )

ax[0,3].pcolormesh(grids["lstm-48"]["xi"], grids["lstm-48"]["yi"], grids["lstm-48"]["zi"],
                   cmap="YlOrRd", vmin=0, vmax=vmax, transform=ccrs.UTM(32),
                  )

ax[1,1].pcolormesh(grids["gat-500"]["xi"], grids["gat-500"]["yi"], grids["gat-500"]["zi"],
                   cmap="YlOrRd", vmin=0, vmax=vmax, transform=ccrs.UTM(32),
                  )

ax[1,2].pcolormesh(grids["gat-vn-500"]["xi"], grids["gat-vn-500"]["yi"], grids["gat-vn-500"]["zi"],
                   cmap="YlOrRd", vmin=0, vmax=vmax, transform=ccrs.UTM(32),
                  )

pcm = ax[1,3].pcolormesh(grids["gat-lstm-vn-6-500"]["xi"], grids["gat-lstm-vn-6-500"]["yi"], grids["gat-lstm-vn-6-500"]["zi"],
                   cmap="YlOrRd", vmin=0, vmax=vmax, transform=ccrs.UTM(32),
                  )

pcb = plt.colorbar(pcm, ax=ax[1, 0], extend="max", orientation="horizontal", shrink=0.8, label="RMSE (m)", anchor=(0.5, 2), panchor=False)
ax[1,0].set_visible(False)


# extent and layout
ax[0,0].set_extent([-1.5, 9.5, 49.5, 56])

[axx.add_feature(cfeature.LAND, facecolor="lightgray", zorder=2) for axx in ax.flatten()]
[axx.add_feature(cfeature.BORDERS, linewidth=0.5, edgecolor="black", zorder=3) for axx in ax.flatten()]
[axx.coastlines(zorder=2, lw=0.5) for axx in ax.flatten()]

ax[0,0].text(0, 1, "A ", transform=ax[0,0].transAxes, ha="right", va="top", fontweight="bold")
ax[0,1].text(0, 1, "B ", transform=ax[0,1].transAxes, ha="right", va="top", fontweight="bold")
ax[0,2].text(0, 1, "C ", transform=ax[0,2].transAxes, ha="right", va="top", fontweight="bold")
ax[0,3].text(0, 1, "D ", transform=ax[0,3].transAxes, ha="right", va="top", fontweight="bold")
ax[1,1].text(0, 1, "E ", transform=ax[1,1].transAxes, ha="right", va="top", fontweight="bold")
ax[1,2].text(0, 1, "F ", transform=ax[1,2].transAxes, ha="right", va="top", fontweight="bold")
ax[1,3].text(0, 1, "G ", transform=ax[1,3].transAxes, ha="right", va="top", fontweight="bold")

ax[0,0].text(.95, 0.05, "Ground truth", transform=ax[0,0].transAxes, ha="right", va="bottom", fontsize=8, bbox={"fc":"w"})
ax[0,1].text(.95, 0.05, "MLP", transform=ax[0,1].transAxes, ha="right", va="bottom", fontsize=8, bbox={"fc":"w"})
ax[0,2].text(.95, 0.05, "LSTM (6 h)", transform=ax[0,2].transAxes, ha="right", va="bottom", fontsize=8, bbox={"fc":"w"})
ax[0,3].text(.95, 0.05, "LSTM (48 h)", transform=ax[0,3].transAxes, ha="right", va="bottom", fontsize=8, bbox={"fc":"w"})
ax[1,1].text(.95, 0.05, "Graph w/o VN", transform=ax[1,1].transAxes, ha="right", va="bottom", fontsize=8, bbox={"fc":"w"})
ax[1,2].text(.95, 0.05, "Graph", transform=ax[1,2].transAxes, ha="right", va="bottom", fontsize=8, bbox={"fc":"w"})
ax[1,3].text(.95, 0.05, "GraphLSTM (6 h)", transform=ax[1,3].transAxes, ha="right", va="bottom", fontsize=8, bbox={"fc":"w"})

plt.subplots_adjust(wspace=0.15, hspace=0.05)

plt.savefig("paper_plots/spatial_rmse_summer_with_ground_truth.png", dpi=220, bbox_inches="tight")

#### Multi-target runs

This section needs revision (TODO)

In [ ]:
skeys = ["lstm-6", "lstm-48", "gat-vn-500", "gat-lstm-vn-6-500"]
slabelkeys = ["LSTM (6 h)", "LSTM (48 h)",  "Graph (500 nodes)", "Graph-LSTM\n(6 h, 500 nodes)"]
sdict = {}

for key, lkey in zip(skeys, slabelkeys):
    sdict[key] = xr.open_dataset(f"./spatial_metrics/multi-{key}_spatial_metrics.nc")

In [ ]:
vmaxs = [0.15, 2.5, 30]

In [ ]:
projection = ccrs.Orthographic(central_longitude=4.5, central_latitude=53.5)

nrows=3
ncols=4

fig, axes_arr = plt.subplots(
            nrows,
            ncols,
            figsize=PAPER_WIDE,
            subplot_kw={"projection": projection},
            squeeze=False,
    )
#axes = axes_arr.ravel().tolist()
fig.subplots_adjust(top=.95, left=0.05, right=.85, bottom=0.05, hspace=0.05, wspace=0.02)

pcm = None
for i, var in enumerate(["sigWaveHeight", "meanWavePeriod", "meanWaveDirection"]):
    axes = axes_arr[i]
    for ax, name in zip(axes, skeys):
        g = sdict[name]
        zi = g.isel(variable=i, season=1)["rmse"]
        pcm = ax.pcolormesh(
                g["x"],
                g["y"],
                zi,
                cmap="YlOrRd",
                vmin=0,
                vmax=vmaxs[i],
                transform=ccrs.UTM(32),
                zorder=1,
        )
        ax.add_feature(cfeature.LAND, facecolor="lightgray", zorder=2)
        ax.add_feature(cfeature.BORDERS, linewidth=0.5, edgecolor="black", zorder=3)
        ax.coastlines(zorder=4, lw=0.5)
        #ax.set_title(name, fontsize=10)
        #ax.text(.95, 0.05, labels[name], transform=ax.transAxes, ha="right", va="bottom", bbox={"color": "white"}, fontsize=9)

    # Single shared colorbar in a dedicated strip on the right.
    cbar_ax = fig.add_axes([0.86, 0.95 - (i+1)*0.3, 0.02, 0.28])

    fig.colorbar(pcm, cax=cbar_ax, ax=None if cbar_ax else axes[:n], label=var, orientation="vertical")

## Rollout

### Read preprocessed data

Rollout was computed for both months of the test set with forecasts of 240 hours (10 days) starting each day of the July 1-20 / December 1-20 at 00:00, 08:00, 16:00. Output metrics are stored in `csv` files that are processed in this section. Note that since LSTM requires input sequences, these rollouts will then produce a later start date. However, the time stamp in the `csv` files corrects for that.



In [ ]:
import glob

In [ ]:
experiments = ["mlp", "lstm-6", "lstm-6-noise-02", "lstm-48", "lstm-48-noise-02", "gat-500", "gat-vn-500", "gat-vn-500-noise-02", "gat-lstm-vn-6-500", "gat-lstm-vn-6-500-noise-02"]

In [ ]:
frames = []

for exp in experiments:
    this_frame = pd.concat([pd.read_csv(f) for f in glob.glob(f"../../logs/rollout/swh-{exp}/rollout_metrics_lead_from_*.csv")])
    this_frame["experiment"] = exp
    frames.append(this_frame)
    
    
all_frames = pd.concat(frames)

In [ ]:
all_frames

Quick check whether all lead times have comparable number of rollouts that reached them

In [ ]:
all_frames["lead"].value_counts().plot();

Number of rollouts should be comparable across experiments

In [ ]:
all_frames["experiment"].value_counts()

In [ ]:
plot_colors = {
    'mlp':                        '#0072B2',
    'lstm-6':                     '#b27b00',
    'lstm-6-noise-02':            '#6b4a00',
    'lstm-48':                    '#ffd066',
    'lstm-48-noise-02':           '#d69400', 
    'gat-500':                    '#D55E00',
    'gat-vn-500':                 '#009E73',
    'gat-vn-500-noise-02':        '#005f45',
    'gat-lstm-vn-6-500':          '#bc508c',
    'gat-lstm-vn-6-500-noise-02': '#713054',
}

In [ ]:
new_rollout_labels = ["MLP", "LSTM (6 h)", "LSTM (48 h)", "Graph (w/o VN) (500 nodes)", "Graph (500 nodes)", "GraphLSTM (6 h, 500 nodes)"]

In [ ]:
lead_times = [6, 24, 48, 96, 144]

### Diagnostic plots

Quick check how the different WindWaveNet architectures perform across lead times, and how the added noise in training affects performance in rollout. Shown first for all predictions, then only for extreme Hs.

In [ ]:
fix, ax = plt.subplots(1, 1, figsize=PAPER_WIDE)

sns.barplot(data=all_frames.query("lead in @lead_times"), x="lead", y="rmse_sigWaveHeight", hue="experiment", palette=plot_colors, linewidth=1, edgecolor="w", errorbar="se")

ax.set_xlabel("Rollout lead time (h)")
ax.set_ylabel("RMSE (m)");

ax.legend(ncols=1, loc="center left", bbox_to_anchor=(1,0.5));

Only bin with Hs > 2 m (too few samples in most extreme > 4 m bin)

In [ ]:
fix, ax = plt.subplots(1, 1, figsize=PAPER_WIDE)

sns.lineplot(data=all_frames, x="lead", y="rmse_sigWaveHeight", hue="experiment", palette=plot_colors, linewidth=1, errorbar=None)

ax.set_xlabel("Rollout lead time (h)")
ax.set_ylabel("RMSE (m)");

ax.legend(ncols=1, loc="center left", bbox_to_anchor=(1,0.5));

In [ ]:
fix, ax = plt.subplots(1, 1, figsize=PAPER_WIDE)

sns.barplot(data=all_frames.dropna(axis=0, subset="rmse_sigWaveHeight_[2.0,4.0)").query("lead in @lead_times"), x="lead", y="rmse_sigWaveHeight_[2.0,4.0)", hue="experiment", palette=plot_colors, linewidth=1, edgecolor="w",)

ax.set_xlabel("Rollout lead time (h)")
ax.set_ylabel("RMSE (m) Hs > 2 m")

ax.legend(ncols=1, loc="center left", bbox_to_anchor=(1,0.5));

### Paper plots

- Top architectures in the main body
- All other architectures in appendix

Version 1: Include all other top architectures as light background lines (not my favorite)

In [ ]:
fix, ax = plt.subplots(1, 3, sharex=True, sharey=True, figsize=PAPER_WIDE)

# LSTM family
sns.lineplot(data=all_frames.query("experiment in ['lstm-6-noise-02', 'gat-vn-500-noise-02', 'gat-lstm-vn-6-500-noise-02']"), x="lead", y="rmse_sigWaveHeight", hue="experiment", palette=plot_colors, linewidth=0.5, alpha=0.7, errorbar=None, legend=True, dashes=False, ax=ax[0])
sns.lineplot(data=all_frames.query("experiment in ['lstm-6', 'lstm-6-noise-02']"), x="lead", y="rmse_sigWaveHeight", hue="experiment", palette=plot_colors, linewidth=1.3, errorbar="se", legend=True, ax=ax[0])
handles = ax[0].get_legend_handles_labels()[0]
ax[0].legend(handles[-2:], ["w/o noise", "with noise"], fontsize=9, loc="lower right", title="LSTM (6 h)")

# Graph-VN family
#sns.lineplot(data=all_frames, x="lead", y="rmse_sigWaveHeight", style="experiment", color="gray", linewidth=0.25, errorbar=None, legend=False, dashes=False, ax=ax[1])
sns.lineplot(data=all_frames.query("experiment in ['lstm-6-noise-02', 'gat-vn-500-noise-02', 'gat-lstm-vn-6-500-noise-02']"), x="lead", y="rmse_sigWaveHeight", hue="experiment", palette=plot_colors, linewidth=0.5, alpha=0.7, errorbar=None, legend=False, dashes=False, ax=ax[1])
sns.lineplot(data=all_frames.query("experiment in ['gat-vn-500', 'gat-vn-500-noise-02']"), x="lead", y="rmse_sigWaveHeight", hue="experiment", palette=plot_colors, linewidth=1.3, errorbar="se", legend=True, ax=ax[1])
handles = ax[1].get_legend_handles_labels()[0]
ax[1].legend(handles, ["w/o noise", "with noise"], fontsize=9, loc="lower right", title="Graph (500 nodes)")

# GraphLSTM-VN family
#sns.lineplot(data=all_frames, x="lead", y="rmse_sigWaveHeight", style="experiment", color="gray", linewidth=0.25, errorbar=None, legend=False, dashes=False, ax=ax[2])
sns.lineplot(data=all_frames.query("experiment in ['lstm-6-noise-02', 'gat-vn-500-noise-02', 'gat-lstm-vn-6-500-noise-02']"), x="lead", y="rmse_sigWaveHeight", hue="experiment", palette=plot_colors, linewidth=0.5, alpha=0.7, errorbar=None, legend=False, dashes=False, ax=ax[2])
sns.lineplot(data=all_frames.query("experiment in ['gat-lstm-vn-6-500', 'gat-lstm-vn-6-500-noise-02']"), x="lead", y="rmse_sigWaveHeight", hue="experiment", palette=plot_colors, linewidth=1.3, errorbar="se", legend=True, ax=ax[2])
handles = ax[2].get_legend_handles_labels()[0]
ax[2].legend(handles, ["w/o noise", "with noise"], fontsize=9, loc="lower right", title="GraphLSTM\n(6 h, 500 nodes)")

[axx.set_xlabel("Rollout lead time (h)") for axx in ax]
ax[0].set_ylabel("RMSE (m)")

[axx.set_xticks([0, 24, 48, 72, 96, 120, 144, 168]) for axx in ax]

#handles = ax.get_legend_handles_labels()[0]
#ax.legend(handles, new_rollout_labels, ncol=1, fontsize=8)

#fig.tight_layout()
plt.subplots_adjust(wspace=0.05)
plt.savefig("paper_plots/rollout_vs_lead_time_with_errorbars_noise-runs.png", bbox_inches="tight", dpi=200)

Version 2: Include only architecture with/without noise.

In [ ]:
fix, ax = plt.subplots(2, 2, sharex=True, sharey=True, figsize=PAPER_WIDE)
ax = ax.flatten()

# LSTM family
sns.lineplot(data=all_frames.query("experiment in ['lstm-6', 'lstm-6-noise-02']"), x="lead", y="rmse_sigWaveHeight", hue="experiment", palette=plot_colors, linewidth=1.3, errorbar="se", legend=True, ax=ax[0])
handles = ax[0].get_legend_handles_labels()[0]
ax[0].legend(handles[-2:], ["w/o noise", "with noise"], fontsize=9, ncols=1, loc="upper left")#, title="LSTM (6 h)")
ax[0].set_title("LSTM (6 h)")

# LSTM 48 family
sns.lineplot(data=all_frames.query("experiment in ['lstm-48', 'lstm-48-noise-02']"), x="lead", y="rmse_sigWaveHeight", hue="experiment", palette=plot_colors, linewidth=1.3, errorbar="se", legend=True, ax=ax[1])
handles = ax[1].get_legend_handles_labels()[0]
ax[1].legend(handles[-2:], ["w/o noise", "with noise"], fontsize=9, ncols=1, loc="lower right")
ax[1].set_title("LSTM (48 h)")

# Graph-VN family
sns.lineplot(data=all_frames.query("experiment in ['gat-vn-500', 'gat-vn-500-noise-02']"), x="lead", y="rmse_sigWaveHeight", hue="experiment", palette=plot_colors, linewidth=1.3, errorbar="se", legend=True, ax=ax[2])
handles = ax[2].get_legend_handles_labels()[0]
ax[2].legend(handles, ["w/o noise", "with noise"], fontsize=9, ncols=1, loc="upper left")
ax[2].set_title("Graph (500 nodes)")

# GraphLSTM-VN family
sns.lineplot(data=all_frames.query("experiment in ['gat-lstm-vn-6-500', 'gat-lstm-vn-6-500-noise-02']"), x="lead", y="rmse_sigWaveHeight", hue="experiment", palette=plot_colors, linewidth=1.3, errorbar="se", legend=True, ax=ax[3])
handles = ax[3].get_legend_handles_labels()[0]
ax[3].legend(handles, ["w/o noise", "with noise"], fontsize=9, ncols=1, loc="upper left")
ax[3].set_title("GraphLSTM (6 h, 500 nodes)")

[axx.set_xlabel("Rollout lead time (h)") for axx in ax]
ax[0].set_ylabel("RMSE (m)")
ax[2].set_ylabel("RMSE (m)")

[axx.set_xticks(np.arange(11)*24) for axx in ax]
[axx.set_xlim(0, 240) for axx in ax]

plt.subplots_adjust(wspace=0.05, hspace=0.2)
plt.savefig("paper_plots/rollout_vs_lead_time_with_errorbars_noise-runs_v2.png", bbox_inches="tight", dpi=200)

In [ ]:
# for appendix
fix, ax = plt.subplots(1, 1, sharex=True, sharey=True, figsize=PAPER_NARROW)

# MLP family
sns.lineplot(data=all_frames.query("experiment in ['gat-lstm-vn-6-500-noise-02']"), x="lead", y="rmse_sigWaveHeight", color="C7", linewidth=0.5, errorbar=None, legend=False, dashes=False, ax=ax)
sns.lineplot(data=all_frames.query("experiment in ['mlp', 'gat-500']"), x="lead", y="rmse_sigWaveHeight", hue="experiment", palette=plot_colors, linewidth=1.3, errorbar="se", legend=True, ax=ax)
handles = ax.get_legend_handles_labels()[0]
ax.legend(handles, ["MLP", "Graph (500 nodes)"])

ax.set_xlabel("Rollout lead time (h)")
ax.set_ylabel("RMSE (m)")

ax.set_xticks(np.arange(11)*24)
ax.set_xlim(0, 240)

#handles = ax.get_legend_handles_labels()[0]
#ax.legend(handles, new_rollout_labels, ncol=1, fontsize=8)

#fig.tight_layout()
plt.savefig("paper_plots/rollout_vs_lead_time_with_errorbars_noise-other-runs_v2.png", bbox_inches="tight", dpi=200)

### Paper metrics

These numbers are used in a table in the manuscript

In [ ]:
lead_times = [6, 24, 96, 168, 240]
all_frames.query("lead in @lead_times").groupby(["lead", "experiment"])["rmse_sigWaveHeight"].mean().round(3)

### Calculate additional rollout metrics

Performance in the last rollout day (> 216 hours) will be used in the section on coupled WindWaveNet. Use standard error (std / sqrt(N)).

In [ ]:
mean_240 = all_frames.query("lead > 216").groupby("experiment")["rmse_sigWaveHeight"].mean()
std_240 = all_frames.query("lead > 216").groupby("experiment")["rmse_sigWaveHeight"].std()
mean_240 = mean_240.reset_index()
mean_240.columns = ["experiment", "swh_rmse_240_mean_m"]
std_240 = std_240.reset_index()
std_240.columns = ["experiment", "swh_rmse_240_std_m"]

In [ ]:
scale = np.sqrt(all_frames.query("lead == 1")["experiment"].value_counts().iloc[0])
scale

In [ ]:
std_240["swh_rmse_240_se_m"] = std_240["swh_rmse_240_std_m"] / scale

In [ ]:
swh_rmse_240_m = pd.merge(mean_240, std_240)
swh_rmse_240_m["noise"] = ["noise" in exp for exp in swh_rmse_240_m.experiment.values]    

In [ ]:
swh_rmse_240_m = swh_rmse_240_m.replace({#"gat-lstm-vn-6-500":"Test_WindWaveNet_Graph-LSTM-VN_Duck",
                                         "gat-lstm-vn-6-500-noise-02":"Test_WindWaveNet_Graph-LSTM-VN_Duck",
                                                     #"gat-vn-500":"Test_WindWaveNet_Graph-VN_Duck",
                                                     "gat-vn-500-noise-02":"Test_WindWaveNet_Graph-VN_Duck",
                                                     #"lstm-48":"Test_WindWaveNet_LSTM-48_Duck",
                                                     "lstm-48-noise-02":"Test_WindWaveNet_LSTM-48_Duck",
                                                     #"lstm-6":"Test_WindWaveNet_LSTM_Duck",
                                                     "lstm-6-noise-02":"Test_WindWaveNet_LSTM_Duck",
                                                     "mlp":"Test_WindWaveNet_MLP_Duck",
                                                     "gat-500":"Test_WindWaveNet_Graph_Duck",
                                                    })
swh_rmse_240_m

## Performance metrics

### Read and combine pre-computed metrics

The performance metrics were calculated in the WWM-Duck test case notebook (coupled runtime) and based on the actual SCHISM-WWM/-WindWaveNet code (memory requirements).

In [ ]:
perfdf = pd.read_csv("performance_metrics.csv") # from WWM-Duck notebook and spreadsheet

In [ ]:
perfdf = pd.merge(perfdf, swh_rmse_240_m, how="left") # add performance metrics from rollout
perfdf.index = perfdf.experiment

In [ ]:
perfdf.loc["Test_WWM_Duck", ["swh_rmse_240_mean_m", "swh_rmse_240_std_m"]] = 0
perfdf

In [ ]:
plot_colors = {
     'Test_WindWaveNet_MLP_Duck':     '#0072B2',
     'Test_WindWaveNet_LSTM_Duck':    '#b27b00',
     'Test_WindWaveNet_LSTM-48_Duck': '#ffd066',
     'Test_WindWaveNet_Graph_Duck':   '#D55E00',
     'Test_WindWaveNet_Graph-VN_Duck': '#009E73',
     #'swh-gat-lstm-vn-6-50': '#81315e', # not used
     'Test_WindWaveNet_Graph-LSTM-VN_Duck':    '#bc508c',
     'Test_WindWaveNet_Graph-LSTM-VN-48_Duck': '#d590b6',
     'Test_WWM_Duck':                          '#000000',
              }

### Diagnostic plots for all models

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=PAPER_NARROW)

#sns.scatterplot(perfdf, x="wall_time_ms", y="swh_rmse_m", size="mem_domain_mb", hue="experiment", palette=plot_colors, legend=False, sizes=(20, 200), alpha=0.5)
sns.scatterplot(perfdf, x="wall_time_ms", y="swh_rmse_m", hue="experiment", palette=plot_colors, legend=False)
sns.scatterplot(perfdf, x="wall_time_ms", y="swh_rmse_m", size="mem_domain_mb", legend=False, sizes=(20, 200), edgecolor="k", facecolor="none")
ax.set_xscale("log")
ax.set_xlabel("CPU wall time (ms)")
ax.set_ylabel("RMSE (m)")
ax.grid(lw=0.5, ls=":")

fig.tight_layout()

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=PAPER_NARROW)

sns.scatterplot(perfdf, x="wall_time_ms", y="swh_rmse_240_mean_m", hue="experiment", palette=plot_colors, legend=False, zorder=2)

ax.errorbar(perfdf["wall_time_ms"], perfdf["swh_rmse_240_mean_m"], yerr=perfdf["swh_rmse_240_se_m"], xerr=perfdf["wall_time_std_ms"], ls="none", elinewidth=0.75, ecolor="C7", fmt="None", capsize=3, zorder=1)

ax.set_xscale("log")
ax.set_xlabel("CPU wall time (ms)")
ax.set_ylabel("RMSE (m)")
ax.grid(lw=0.5, ls=":")

fig.tight_layout()

### Paper plot

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=PAPER_NARROW)

keep = ["Test_WindWaveNet_LSTM_Duck", "Test_WindWaveNet_LSTM-48_Duck", "Test_WindWaveNet_Graph-VN_Duck", "Test_WindWaveNet_Graph-LSTM-VN_Duck"]

sns.scatterplot(perfdf.query("experiment in @keep"), x="wall_time_ms", y="swh_rmse_240_mean_m", hue="experiment", style="experiment", palette=plot_colors, legend=True, s=60,  zorder=2)

ax.errorbar(perfdf.query("experiment in @keep")["wall_time_ms"], 
            perfdf.query("experiment in @keep")["swh_rmse_240_mean_m"], 
            yerr=perfdf.query("experiment in @keep")["swh_rmse_240_se_m"], 
            xerr=perfdf.query("experiment in @keep")["wall_time_std_ms"], 
            ls="none", elinewidth=0.75, ecolor="C7", fmt="None", capsize=3, zorder=1)

handles = ax.get_legend_handles_labels()[0]
ax.legend(handles, ["LSTM (6 h)", "LSTM (48 h)", "Graph", "GraphLSTM (6 h)"], ncols=1, fontsize=8, loc=2)

ax.vlines(perfdf.query("experiment == 'Test_WWM_Duck'")["wall_time_ms"].values[0], ymin=0.1, ymax=0.325, lw=1.8, ls="--", color="k")
ax.text(220, 0.1, "WWM reference", ha="right", va="bottom", fontsize=8, transform=ax.transData)

ax.set_xscale("log")
ax.set_xlim(10, 300)
ax.set_xticks([10, 20, 50, 60, 70, 80, 90, 100, 200])
ax.set_xticklabels([10, 20, 50, "", "", "", "", 100, 200])
ax.set_xlabel("CPU wall time / call (ms)")
ax.set_ylabel("RMSE on Day 10 (m)")
ax.grid(lw=0.5, ls=":")



fig.tight_layout()
plt.savefig('paper_plots/performance_coupled_rollout_240_best.png', dpi=200, bbox_inches="tight")

## Buoys

### Setup

Read buoy data and ground truth

In [ ]:
from pyproj import Transformer
DATA_CRS = ccrs.UTM(32)  # SCHISM_hgrid_node_x/y are projected UTM zone 32N
LONLAT_TRANSFORMER = Transformer.from_crs("epsg:32632", "epsg:4326", always_xy=True)
import re

In [ ]:
ds_true = xr.open_dataset("/work/ka1176/shared_data/2025-wwm-emulator-schism/processed_RUN04b_JUWELS_wpile.zarr/")
ds_true = ds_true.sel(time=slice("2020-07-01", "2020-07-31")) # only summer month available
simt = ds_true.time # simulation time stamps
nlon, nlat = LONLAT_TRANSFORMER.transform(ds_true.SCHISM_hgrid_node_x.values, ds_true.SCHISM_hgrid_node_y.values)

In [ ]:
nlat[:10], nlon[:10], nlon[-10:]

In [ ]:
buoy_dict = {
    "WES": {"path": "/work/bg1581/g260200/buoy/wavespec_2020/WES/wavespec_202007_WES.nc", "nSCHISM_hgrid_node": None, "buoy_lonlat": None},
    "HEL": {"path": "/work/bg1581/g260200/buoy/wavespec_2020/HEL/wavespec_202007_HEL.nc", "nSCHISM_hgrid_node": None, "buoy_lonlat": None},
    "ELB": {"path": "/work/bg1581/g260200/buoy/wavespec_2020/ELB/wavespec_202007_ELB.nc", "nSCHISM_hgrid_node": None, "buoy_lonlat": None},
    "FN3": {"path": "/work/bg1581/g260200/buoy/wavespec_2020/FN3/wavespec_202007_FN3.nc", "nSCHISM_hgrid_node": None, "buoy_lonlat": None},
}

In [ ]:
for key, val in buoy_dict.items():
    with xr.open_dataset(val["path"]) as bds:
        attr = bds.attrs
        val["buoy_lonlat"] = (float(re.findall(r"\d+.\d+", attr["StartLongitude"])[0]), float(re.findall(r"\d+.\d+", attr["StartLatitude"])[0]))
        distance = np.sqrt( (nlon - val["buoy_lonlat"][0])**2 + (nlat - val["buoy_lonlat"][1])**2  )
        val["nSCHISM_hgrid_node"] = np.argmin(distance)

In [ ]:
buoy_dict

### Diagnostic plots

Check whether the locations selected by matching buoy location against SCHISM grid nodes do actually align with where they should be

In [ ]:
fig, ax = plt.subplots(1, 1, subplot_kw={"projection": ccrs.PlateCarree()})

ax.set_extent([7, 9, 53.5, 55.5])

for i, (key, val) in enumerate(buoy_dict.items()):
    ax.scatter(val["buoy_lonlat"][0], val["buoy_lonlat"][1], label=key, color=f"C{i}", facecolor="none")
    ax.scatter(nlon[val["nSCHISM_hgrid_node"]], nlat[val["nSCHISM_hgrid_node"]], color=f"C{i}", marker="*")

ax.add_feature(cfeature.LAND, facecolor="#e8e8e8", zorder=1)
ax.coastlines(resolution="10m", linewidth=0.6, zorder=3)

ax.legend(loc=3);

In [ ]:
buoy_experiments = ["mlp", "lstm-6", "lstm-48", #"gat-500", 
                    "gat-vn-500", "gat-lstm-vn-6-500"]

In [ ]:
plot_colors = {
    'mlp':                        '#0072B2',
    'lstm-6':                     '#b27b00',
    'lstm-6-noise-02':            '#6b4a00',
    'lstm-48':                    '#ffd066',
    'lstm-48-noise-02':           '#d69400', 
    'gat-500':                    '#D55E00',
    'gat-vn-500':                 '#009E73',
    'gat-vn-500-noise-02':        '#005f45',
    'gat-lstm-vn-6-500':          '#bc508c',
    'gat-lstm-vn-6-500-noise-02': '#713054',
}

In [ ]:
fig, ax = plt.subplots(8, 1, sharex=True, figsize=(12, 12))

for i, (key, val) in enumerate(buoy_dict.items()):
    with xr.open_dataset(val["path"]) as ds:
        hs = ds["hs"]
        bt = ds["time"]

    ax[2*i].plot(bt, hs, color="C7", ls="--", label="Buoy")
    
    # Ground truth
    swh = ds_true.isel(nSCHISM_hgrid_node=val["nSCHISM_hgrid_node"])["sigWaveHeight"]#.values
    ax[2*i].plot(simt, swh, color="k", ls="-", lw=2, label="Ground truth")

    # predictions
    for exp in buoy_experiments:
        with xr.open_dataset(f"/work/ka1176/caroline/gitlab/wind-wave-model-emulator/logs/rollout-buoy/swh-{exp}/rollout_predictions.nc") as wdf:
            wswh = wdf.isel(nSCHISM_hgrid_node=val["nSCHISM_hgrid_node"])["sigWaveHeight"]#.values
            wtime = wdf.time
        joint = xr.concat([swh, wswh], dim="experiment", join="inner")

        jointobs = xr.concat([wswh, hs.resample(time="1h").mean()], dim="experiment", join="inner").dropna(dim="time")

        ax[2*i+1].plot(joint.time, joint.isel(experiment=1) - joint.isel(experiment=0), label=exp)
    
    ax[2*i].text(0, 1, key, transform=ax[2*i].transAxes, fontsize=14, ha="left", va="top")

    #ax[i].legend()

In [ ]:
# winter
fig, ax = plt.subplots(8, 1, sharex=True, figsize=(12, 12))

ds_true_winter = xr.open_dataset("/work/ka1176/shared_data/2025-wwm-emulator-schism/processed_RUN04b_JUWELS_wpile.zarr/")
ds_true_winter = ds_true_winter.sel(time=slice("2020-12-01", None)) # winter
simt_winter = ds_true_winter.time # simulation time stamps

for i, (key, val) in enumerate(buoy_dict.items()):
    #with xr.open_dataset(val["path"]) as ds:
    #    hs = ds["hs"]
    #    bt = ds["time"]

    #ax[2*i].plot(bt, hs, color="C7", ls="--", label="Buoy")
    
    # Ground truth
    swh = ds_true_winter.isel(nSCHISM_hgrid_node=val["nSCHISM_hgrid_node"])["sigWaveHeight"]#.values
    ax[2*i].plot(simt_winter, swh, color="k", ls="-", lw=2, label="Ground truth")

    # predictions
    for exp in buoy_experiments:
        with xr.open_dataset(f"/work/ka1176/caroline/gitlab/wind-wave-model-emulator/logs/rollout-buoy/swh-{exp}/rollout_predictions_from_336_0.nc") as wdf:
            wswh = wdf.isel(nSCHISM_hgrid_node=val["nSCHISM_hgrid_node"])["sigWaveHeight"]#.values
            wtime = wdf.time
        joint = xr.concat([swh, wswh], dim="experiment", join="inner")

        ax[2*i].plot(joint.time, joint.isel(experiment=1), label=exp)

        ax[2*i+1].plot(joint.time, joint.isel(experiment=1) - joint.isel(experiment=0), label=exp)
    
    ax[2*i].text(0, 1, key, transform=ax[2*i].transAxes, fontsize=14, ha="left", va="top")

    #ax[i].legend()

### Metrics

Calculate RMSE of WindWaveNet rollout against model (not buoy observations!) at the selected buoy locations

In [ ]:
for i, (key, val) in enumerate(buoy_dict.items()):
    with xr.open_dataset(val["path"]) as ds:
        hs = ds["hs"]
        bt = ds["time"]

    print(key)
    
    # Ground truth
    swh = ds_true.isel(nSCHISM_hgrid_node=val["nSCHISM_hgrid_node"])["sigWaveHeight"]#.values

    # predictions
    for j, exp in enumerate(["lstm-6", "lstm-48", "gat-vn-500", "gat-lstm-vn-6-500"]):
        if True:
            with xr.open_dataset(f"/work/ka1176/caroline/gitlab/wind-wave-model-emulator/logs/rollout-buoy/swh-{exp}/rollout_predictions.nc") as wdf:
                wswh = wdf.isel(nSCHISM_hgrid_node=val["nSCHISM_hgrid_node"])["sigWaveHeight"]#.values
                wtime = wdf.time

            with xr.open_dataset(f"/work/ka1176/caroline/gitlab/wind-wave-model-emulator/logs/rollout-buoy/swh-{exp}-noise/rollout_predictions_from_183_0.nc") as wdf:
                wswh_noise = wdf.isel(nSCHISM_hgrid_node=val["nSCHISM_hgrid_node"])["sigWaveHeight"]#.values
                wtime = wdf.time

            joint1 = xr.concat([swh, wswh], dim="experiment", join="inner")
            joint2 = xr.concat([swh, wswh_noise], dim="experiment", join="inner")

            rmse = root_mean_squared_error(joint1.isel(experiment=0).values, joint1.isel(experiment=1).values)
            print(f"{exp}        {rmse:.3f}")
            rmse = root_mean_squared_error(joint2.isel(experiment=0).values, joint2.isel(experiment=1).values)
            print(f"{exp} noise  {rmse:.3f}")


### Paper plots

In [ ]:
fig, ax = plt.subplots(4, 1, sharex=True, sharey=True, figsize=PAPER_WIDE)

linestyles = ["-", "-"]
#linestyles = ["--", "-."]

for i, (key, val) in enumerate(buoy_dict.items()):
    with xr.open_dataset(val["path"]) as ds:
        hs = ds["hs"]
        bt = ds["time"]

    ax[i].plot(bt, hs, color="C7", ls=":", label="Observation")
    
    # Ground truth
    swh = ds_true.isel(nSCHISM_hgrid_node=val["nSCHISM_hgrid_node"])["sigWaveHeight"]#.values
    ax[i].plot(simt, swh, color="k", ls="-", lw=1, label="SCHISM-WWM")

    # predictions
    for j, exp in enumerate(["lstm-6", "gat-lstm-vn-6-500"]):
        if exp == "lstm-6" or exp == "gat-lstm-vn-6-500":

            with xr.open_dataset(f"/work/ka1176/caroline/gitlab/wind-wave-model-emulator/logs/rollout-buoy/swh-{exp}-noise/rollout_predictions_from_183_0.nc") as wdf:
                wswh_noise = wdf.isel(nSCHISM_hgrid_node=val["nSCHISM_hgrid_node"])["sigWaveHeight"]#.values
                wtime = wdf.time

            ax[i].plot(wtime, wswh_noise, label=exp, color=plot_colors[exp], ls=linestyles[j], zorder=4)
    
    ax[i].text(0.5, 0.95, key, transform=ax[i].transAxes, fontsize=9, ha="center", va="top")

handles, _ = ax[0].get_legend_handles_labels()
fig.legend(handles, ["Observation", "SCHISM-WWM", "LSTM (6 h) with noise", "GraphLSTM (6 h, 500 nodes) with noise"], loc="upper left", bbox_to_anchor=(0.15, 1.0),
           ncol=2, frameon=False, fontsize=8)

ax[3].set_ylim(0, 4)
ax[3].set_yticks([0, 1, 2, 3, 4])
ax[3].set_yticklabels([0, 1, 2, 3, 4])

ax[3].set_xlabel("July 2020")
ax[3].set_xlabel("July 2020")

ax[3].set_xticks(list(range(18444, 18444+31))[::3])
ax[3].set_xticklabels(list(range(1, 32))[::3])

fig.text(-0.1, -0.2, "Significant wave height (m)", rotation=90, transform=ax[2].transAxes, ha="center")
plt.subplots_adjust(wspace=0.05)

#fig.tight_layout()
plt.savefig("paper_plots/buoy_selected_models_noise.png", dpi=200, bbox_inches="tight")

In [ ]:
fig, ax = plt.subplots(4, 1, sharex=True, sharey=True, figsize=PAPER_WIDE)

for i, (key, val) in enumerate(buoy_dict.items()):
    with xr.open_dataset(val["path"]) as ds:
        hs = ds["hs"]
        bt = ds["time"]

    ax[i].plot(bt, hs, color="C7", ls=":", label="Observation")
    
    # Ground truth
    swh = ds_true.isel(nSCHISM_hgrid_node=val["nSCHISM_hgrid_node"])["sigWaveHeight"]#.values
    ax[i].plot(simt, swh, color="k", ls="-", lw=1.2, label="SCHISM-WWM")

    this_r2_score = {}

    # predictions
    for exp in buoy_experiments:
        if exp != "lstm-6" and exp != "gat-lstm-vn-6-500" and exp != "mlp":
            with xr.open_dataset(f"/work/ka1176/caroline/gitlab/wind-wave-model-emulator/logs/rollout-buoy/swh-{exp}-noise/rollout_predictions_from_183_0.nc") as wdf:
                wswh = wdf.isel(nSCHISM_hgrid_node=val["nSCHISM_hgrid_node"])["sigWaveHeight"]#.values
                wtime = wdf.time

            ax[i].plot(wtime, wswh, label=exp, color=plot_colors[exp])            
    
    ax[i].text(0.5, 0.95, key, transform=ax[i].transAxes, fontsize=9, ha="center", va="top")

handles, _ = ax[0].get_legend_handles_labels()
fig.legend(handles, ["Observation", "SCHISM-WWM", "LSTM (48 h) with noise", "Graph (500 nodes) with noise"], loc="upper center",# bbox_to_anchor=(0.5, 1.05),
           ncol=2, frameon=False, fontsize=9)

ax[3].set_ylim(0, 4)
ax[3].set_yticks([0, 1, 2, 3, 4])

ax[3].set_xlabel("Day in July 2020")

ax[3].set_xticks(list(range(18444, 18444+31))[::3])
ax[3].set_xticklabels(list(range(1, 32))[::3])

fig.text(-0.075, -0.2, "Significant wave height (m)", rotation=90, transform=ax[2].transAxes, ha="center")

#fig.tight_layout()
plt.savefig("paper_plots/buoy_appendix_other_models.png", dpi=200, bbox_inches="tight")

In [ ]:
# todo: calculate r2 vs buoy and model for the different architectures